# Advanced: Multi-Strategy Name Generation with PAMOLA.CORE

**Goal**: Master all 3 name generation strategies using operation.execute()

**What you'll learn:**
- **Strategy 1**: English names with gender-aware generation
- **Strategy 2**: Multi-language names (Vietnamese, Russian)
- **Strategy 3**: Complex formats with Faker library
- Calculate advanced privacy metrics
- Analyze name distributions and gender consistency
- Production deployment patterns

**Prerequisites:**
- Completed the simple notebook
- Understanding of name privacy concepts
- Familiarity with operation.execute() pattern
- Python 3.8+
- PAMOLA.CORE installed (auto-installs if needed)

---

## Step 1: Setup and Imports

**How to use:**
- Run this cell to import required libraries
- Auto-detects PAMOLA installation path (searches up 6 levels)
- If not found locally, auto-installs from GitHub

**What you'll see:**
- Detection message showing PAMOLA location or installation progress
- Import confirmation (✅ All imports successful!)
- Notebook start timestamp
- Project root and working directory paths

**Note:** First run may take 1-2 minutes if installing from GitHub

**Expected structure:**
```
PAMOLA/
├── pamola_core/          ← Auto-detected
└── examples/
    ├── data_examples/    ← Data directory
    └── fake_data/name/
        └── 02_fake_name_advanced.ipynb  ← You are here
```

In [ ]:
import pandas as pd
import numpy as np
import json
import sys
import os
from pathlib import Path
from datetime import datetime
import time
from IPython.display import Image, display
 
print("🔍 Detecting PAMOLA installation...\n") 
 
# Auto-detect project root (search up 6 levels for pamola_core/) 
current_dir = Path.cwd() 
project_root = current_dir 
pamola_found = False

for level in range(6): 
    if (project_root / 'pamola_core').exists(): 
        print(f"✓ Found pamola_core at level {level}: {project_root}") 
        sys.path.insert(0, str(project_root))
        pamola_found = True
        break 
    project_root = project_root.parent 

# If not found locally, try to install from Git
if not pamola_found:
    print("⚠️  pamola_core not found in project structure")
    print("📦 Attempting to install from GitHub...\n")
    
    try:
        import subprocess
        
        install_cmd = [
            sys.executable, "-m", "pip", "install",
            "git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        ]
        
        result = subprocess.run(
            install_cmd,
            capture_output=True,
            text=True,
            check=True
        )
        
        print("✅ Successfully installed pamola_core from GitHub!")
        print("   Repository: https://github.com/DGT-Network/PAMOLA.git")
        print("   Branch: pre-epic3")
        
    except subprocess.CalledProcessError as e:
        print(f"❌ Installation failed!")
        print(f"   Error: {e.stderr}")
        raise RuntimeError(
            "Could not find pamola_core locally and installation from GitHub failed. "
            "Please install manually using:\n"
            "pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        )
    except Exception as e:
        print(f"❌ Unexpected error during installation: {e}")
        raise

# Now try to import the required modules
try:
    from pamola_core.fake_data.operations.name_op import FakeNameOperation
    from pamola_core.utils.ops.op_data_source import DataSource
    from pamola_core.utils.progress import HierarchicalProgressTracker
    from pamola_core.utils.tasks.task_reporting import TaskReporter
    from pamola_core.io.csv import read_csv
    
    print("✅ All imports successful!")
    print(f"📅 Notebook started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 80)
    print(f"📂 Project root:  {project_root.name}")
    print(f"📂 Working dir:   {current_dir.relative_to(project_root)}")
    print("=" * 80)
    
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print("\n💡 Troubleshooting:")
    print("   1. Try restarting your Python kernel/notebook")
    print("   2. Verify installation: pip list | grep pamola")
    print("   3. Manual install: pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3")
    raise

## Step 2: Load Complex Dataset

**How to use:**
- Run to load or generate 1000-record dataset
- Auto-creates sample if file not found
- Review data structure before proceeding

**What you'll see:**
- Load status (from file or generated)
- Dataset overview (1000 records, 7 columns)
- Sample records (first 5 rows)
- Column statistics (types, unique counts)

**Note:** Generated data auto-saves to examples/data_examples/ for reuse

In [ ]:
# Try to load from file first (in examples directory)
examples_dir = project_root / 'examples'
data_path = examples_dir / 'data_examples' / 'sample.csv'
print(f"📂 Looking for data at: {data_path}\n")

if data_path.exists():
    print(f"📂 Loading data from: {data_path}")
    df = read_csv(data_path)
    print(f"✓ Loaded {len(df)} records from file")
else:
    print("📊 Generating synthetic dataset (1000 records)...\n")
    
    np.random.seed(42)
    n_records = 1000
    
    # Generate realistic full names with diversity
    first_names_male = [
        'James', 'John', 'Robert', 'Michael', 'William', 'David', 'Richard', 'Joseph',
        'Thomas', 'Charles', 'Christopher', 'Daniel', 'Matthew', 'Anthony', 'Mark',
        'Donald', 'Steven', 'Paul', 'Andrew', 'Joshua', 'Kenneth', 'Kevin', 'Brian',
        'George', 'Timothy', 'Ronald', 'Edward', 'Jason', 'Jeffrey', 'Ryan'
    ]
    
    first_names_female = [
        'Mary', 'Patricia', 'Jennifer', 'Linda', 'Barbara', 'Elizabeth', 'Susan',
        'Jessica', 'Sarah', 'Karen', 'Nancy', 'Lisa', 'Betty', 'Margaret', 'Sandra',
        'Ashley', 'Kimberly', 'Emily', 'Donna', 'Michelle', 'Carol', 'Amanda',
        'Melissa', 'Deborah', 'Stephanie', 'Dorothy', 'Rebecca', 'Sharon', 'Laura'
    ]
    
    last_names = [
        'Smith', 'Johnson', 'Williams', 'Brown', 'Jones', 'Garcia', 'Miller', 'Davis',
        'Rodriguez', 'Martinez', 'Hernandez', 'Lopez', 'Gonzalez', 'Wilson', 'Anderson',
        'Thomas', 'Taylor', 'Moore', 'Jackson', 'Martin', 'Lee', 'Perez', 'Thompson',
        'White', 'Harris', 'Sanchez', 'Clark', 'Ramirez', 'Lewis', 'Robinson'
    ]
    
    countries = ['US', 'VN', 'RU']
    departments = ['Engineering', 'Sales', 'Marketing', 'HR', 'Finance', 'Operations']
    
    # Generate gender distribution
    genders = np.random.choice(['M', 'F'], n_records, p=[0.5, 0.5])
    
    # Generate names based on gender
    full_names = []
    for gender in genders:
        if gender == 'M':
            first = np.random.choice(first_names_male)
        else:
            first = np.random.choice(first_names_female)
        last = np.random.choice(last_names)
        full_names.append(f"{first} {last}")
    
    # Generate dataset
    df = pd.DataFrame({
        'employee_id': [f'EMP{i:05d}' for i in range(1, n_records + 1)],
        'full_name': full_names,
        'gender': genders,
        'country': np.random.choice(countries, n_records, p=[0.7, 0.2, 0.1]),
        'department': np.random.choice(departments, n_records),
        'age': np.random.randint(22, 65, n_records),
        'hire_year': np.random.randint(2010, 2024, n_records)
    })
    
    # Save for future use (with error handling)
    try:
        data_path.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(data_path, index=False)
        print(f"✓ Generated and saved to: {data_path}")
    except PermissionError:
        print(f"⚠️  Cannot save to {data_path} (file may be open)")
        print(f"   Dataset generated in memory only")

print(f"\n📊 Dataset Overview:")
print("=" * 80)
print(f"  Records: {len(df):,}")
print(f"  Columns: {', '.join(df.columns)}")

print(f"\n🔍 Sample Records:")
print(df.head())

print(f"\n📈 Column Statistics:")
for col in df.columns:
    dtype_str = str(df[col].dtype)
    if pd.api.types.is_string_dtype(df[col]):
        print(f"  {col:15s} ({dtype_str:10s}): {df[col].nunique()} unique values")
    else:
        print(f"  {col:15s} ({dtype_str:10s}): min={df[col].min()}, max={df[col].max()}")

print(f"\n📊 Gender Distribution:")
gender_counts = df['gender'].value_counts()
for gender, count in gender_counts.items():
    print(f"  {gender}: {count} ({count/len(df)*100:.1f}%)")

print(f"\n💡 Perfect dataset for testing all 3 strategies!")

## Step 3: Setup Shared Environment

**How to use:**
1. **CUSTOMIZE FIELD_CONFIG**: Edit field names for name generation
   - `"name_field"`: Column containing full names
   - `"gender_field"`: Column with gender information
   - `"country_field"`: Column with country/language code
2. Run to validate fields and setup environment

**What you'll see:**
- Field validation (✓ marks for each field)
- Unique value counts per field
- Task directory created (✓)
- TaskReporter initialized (✓)
- DataSource and config ready (✓)

**Note:** All fields must exist in dataset before executing strategies

In [ ]:
# Field configuration - CUSTOMIZE THESE!
FIELD_CONFIG = {
    "name_field": "full_name",     # Field containing full names
    "gender_field": "gender",      # Field with gender information
    "country_field": "country"     # Field with country/language code
}

# Validate all fields exist in dataset
print("📋 Validating field configuration...\n")
for strategy, field_name in FIELD_CONFIG.items():
    if field_name not in df.columns:
        raise ValueError(
            f"❌ Field '{field_name}' for {strategy} not found!\n"
            f"Available columns: {', '.join(df.columns)}\n"
            f"Please update FIELD_CONFIG"
        )
    print(f"  ✓ {strategy:20s}: '{field_name}' ({df[field_name].nunique()} unique values)")

# Configuration helper (production pattern)
def create_config_kwargs():
    return {
        "dataset_name": "main_dataset"
    }

# Create task directory (in examples/data_examples)
examples_dir = project_root / 'examples'
task_dir = examples_dir / 'data_examples' / 'advanced_output'
os.makedirs(task_dir, exist_ok=True)
print(f"\n✓ Task directory: {task_dir}")

# Initialize TaskReporter
task_reporter = TaskReporter(
    task_id="advanced_name_001",
    task_type="multi_strategy_name",
    description="Multi-strategy name generation",
    report_path=task_dir
)
print("✓ TaskReporter initialized")

# Get config
kwargs = create_config_kwargs()
print(f"✓ Config kwargs ready")

# Create DataSource
data_source = DataSource(
    dataframes={"main_dataset": df}
)
print("✓ DataSource created")

print(f"\n✅ Shared environment ready for all strategies!")

## STRATEGY 1: English Names with Gender-Aware Generation

**How to use:**
- Uses English name dictionary with gender consistency
- Standard FirstName LastName (FL) format
- Run to execute gender-aware strategy

**Key parameters:**
- `language='en'`: English name dictionary
- `gender_field='gender'`: Use gender column
- `format='FL'`: FirstName LastName pattern
- `f_m_ratio=0.5`: Balanced male/female ratio
- `mode='ENRICH'`: Keeps original + adds generated field

**What you'll see:**
- Configuration summary
- Progress: validation → processing → metrics → viz → save
- Completion time and status
- Name examples with gender consistency

**Note:** Creates `full_name_en_gender` with appropriate male/female names

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 1: ENGLISH GENDER-AWARE NAMES")
print("=" * 80 + "\n")

# Setup progress tracker
tracker_s1 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 1: Gender-Aware",
    unit="steps",
    track_memory=True,
    level=0
)

# Create operation
operation_s1 = FakeNameOperation(
    # Core parameters
    field_name=FIELD_CONFIG["name_field"],
    mode='ENRICH',
    
    # Output configuration
    output_field_name=f"{FIELD_CONFIG['name_field']}_en_gender",
    output_format='csv',
    
    # Name generation
    language='en',
    gender_field=FIELD_CONFIG["gender_field"],
    gender_from_name=False,
    format='FL',  # FirstName LastName
    f_m_ratio=0.5,
    use_faker=False,
    case='title',
    
    # Consistency
    consistency_mechanism='mapping',
    key='strategy1-key',
    save_mapping=True,
    id_field='resume_id',
    
    # Processing settings
    use_vectorization=True,
    parallel_processes=2,
    use_cache=False,
    
    # Execution behavior
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 1 configured")
print(f"\n⚙️  Executing...\n")
start_time = time.time()

# Execute
result_s1 = operation_s1.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy1_en_gender',
    reporter=task_reporter,
    progress_tracker=tracker_s1,
    **kwargs
)

elapsed_s1 = time.time() - start_time
print(f"\n✅ Strategy 1 completed in {elapsed_s1:.2f}s")

# Load results (NEWEST file)
output_files_s1 = sorted(
    list((task_dir / 'strategy1_en_gender' / 'output').glob('*.csv')),
    key=lambda x: x.stat().st_mtime, reverse=True
)
if output_files_s1:
    result_df_s1 = pd.read_csv(output_files_s1[0])
    field_name = FIELD_CONFIG["name_field"]
    output_field_s1 = f"{field_name}_en_gender"
    gender_field = FIELD_CONFIG["gender_field"]
    
    print(f"\n🎭 Sample Generated Names (with Gender):")
    print(f"\n{'Original Name':<25} → {'Generated Name':<25} {'Gender':<8}")
    print("-" * 60)
    for i in range(min(10, len(result_df_s1))):
        orig = result_df_s1[field_name].iloc[i]
        gen = result_df_s1[output_field_s1].iloc[i]
        gender = result_df_s1[gender_field].iloc[i]
        print(f"{orig:<25} → {gen:<25} {gender:<8}")
    
    # Gender consistency check
    print(f"\n📊 Gender Distribution:")
    gender_dist = result_df_s1[gender_field].value_counts()
    for g, count in gender_dist.items():
        print(f"  {g}: {count} ({count/len(result_df_s1)*100:.1f}%)")

## STRATEGY 2: Multi-Language Names (Vietnamese/Russian)

**How to use:**
- Demonstrates Vietnamese and Russian name generation
- Language-specific name dictionaries
- Run to execute multi-language strategy

**Key parameters:**
- `language='vi'`: Vietnamese names (can also use 'ru' for Russian)
- `format='LF'`: LastName FirstName (Vietnamese convention)
- `gender_field=None`: No gender-based filtering
- `case='title'`: Proper case formatting

**What you'll see:**
- Configuration summary
- Progress: validation → processing → metrics → viz → save
- Completion time and status
- Vietnamese/Russian name examples

**Note:** Best for multilingual datasets with country-specific requirements

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 2: MULTI-LANGUAGE NAMES")
print("=" * 80 + "\n")

tracker_s2 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 2: Multi-Language",
    unit="steps",
    track_memory=True,
    level=0
)

operation_s2 = FakeNameOperation(
    field_name=FIELD_CONFIG["name_field"],
    mode='ENRICH',
    output_field_name=f"{FIELD_CONFIG['name_field']}_vi",
    output_format='csv',
    
    # Name generation - Vietnamese
    language='vi',  # Vietnamese names
    gender_field=None,  # No gender filtering
    gender_from_name=False,
    format='LF',  # LastName FirstName (Vietnamese convention)
    use_faker=False,
    case='title',
    
    # Consistency
    consistency_mechanism='mapping',
    key='strategy2-key',
    save_mapping=True,
    id_field='resume_id',
    
    use_vectorization=True,
    parallel_processes=2,
    use_cache=False,
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 2 configured")
start_time = time.time()

result_s2 = operation_s2.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy2_multilang',
    reporter=task_reporter,
    progress_tracker=tracker_s2,
    **kwargs
)

elapsed_s2 = time.time() - start_time
print(f"\n✅ Strategy 2 completed in {elapsed_s2:.2f}s")

# Load results
output_files_s2 = sorted(
    list((task_dir / 'strategy2_multilang' / 'output').glob('*.csv')),
    key=lambda x: x.stat().st_mtime, reverse=True
)
if output_files_s2:
    result_df_s2 = pd.read_csv(output_files_s2[0])
    output_field_s2 = f"{FIELD_CONFIG['name_field']}_vi"
    
    print(f"\n🌏 Sample Vietnamese Names:")
    print(f"\n{'Original Name':<25} → {'Vietnamese Name':<25}")
    print("-" * 52)
    for i in range(min(10, len(result_df_s2))):
        orig = result_df_s2[FIELD_CONFIG['name_field']].iloc[i]
        gen = result_df_s2[output_field_s2].iloc[i]
        print(f"{orig:<25} → {gen:<25}")
    
    # Name length analysis
    orig_len = result_df_s2[FIELD_CONFIG['name_field']].str.len().mean()
    gen_len = result_df_s2[output_field_s2].str.len().mean()
    print(f"\n📏 Average Name Length:")
    print(f"  Original: {orig_len:.1f} chars")
    print(f"  Generated: {gen_len:.1f} chars")

## STRATEGY 3: Complex Formats with Faker Library

**How to use:**
- Uses Faker library for maximum name diversity
- Complex FML format (FirstName MiddleName LastName)
- Run to execute Faker-based strategy

**Key parameters:**
- `use_faker=True`: Use Faker library
- `format='FML'`: First Middle Last pattern
- `language='en'`: English locale in Faker
- `gender_from_name=True`: Infer gender from generated names

**What you'll see:**
- Configuration summary
- Progress: validation → processing → metrics → viz → save
- Completion time and status
- Complex name examples with middle names

**Note:** Optimal for maximum realism and name variety

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 3: FAKER LIBRARY WITH MIDDLE NAMES")
print("=" * 80 + "\n")

tracker_s3 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 3: Faker",
    unit="steps",
    track_memory=True,
    level=0
)

operation_s3 = FakeNameOperation(
    field_name=FIELD_CONFIG["name_field"],
    mode='ENRICH',
    output_field_name=f"{FIELD_CONFIG['name_field']}_faker",
    output_format='csv',
    
    # Name generation - Faker
    language='en',
    use_faker=True,  # Use Faker library
    format='FML',  # First Middle Last
    gender_field=None,
    gender_from_name=True,  # Infer from generated name
    case='title',
    
    # Consistency
    consistency_mechanism='mapping',
    key='strategy3-key',
    save_mapping=True,
    id_field='resume_id',
    
    use_vectorization=True,
    parallel_processes=2,
    use_cache=False,
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 3 configured")
start_time = time.time()

result_s3 = operation_s3.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy3_faker',
    reporter=task_reporter,
    progress_tracker=tracker_s3,
    **kwargs
)

elapsed_s3 = time.time() - start_time
print(f"\n✅ Strategy 3 completed in {elapsed_s3:.2f}s")

# Load results
output_files_s3 = sorted(
    list((task_dir / 'strategy3_faker' / 'output').glob('*.csv')),
    key=lambda x: x.stat().st_mtime, reverse=True
)
if output_files_s3:
    result_df_s3 = pd.read_csv(output_files_s3[0])
    output_field_s3 = f"{FIELD_CONFIG['name_field']}_faker"
    
    print(f"\n📚 Sample Complex Names (with Middle Names):")
    print(f"\n{'Original Name':<25} → {'Generated Name (FML)':<35}")
    print("-" * 62)
    for i in range(min(10, len(result_df_s3))):
        orig = result_df_s3[FIELD_CONFIG['name_field']].iloc[i]
        gen = result_df_s3[output_field_s3].iloc[i]
        print(f"{orig:<25} → {gen:<35}")
    
    # Analyze name complexity
    name_parts = result_df_s3[output_field_s3].str.split().str.len()
    print(f"\n📊 Name Complexity:")
    print(f"  Average parts: {name_parts.mean():.1f}")
    print(f"  3-part names: {(name_parts == 3).sum()} ({(name_parts == 3).sum()/len(result_df_s3)*100:.1f}%)")

## Step 5: Strategy Comparison

**How to use:**
- Run AFTER all strategies complete
- Review performance and name characteristics

**What you'll see:**
- Execution times (fastest to slowest)
- Total processing time
- Name format comparison
- Language/generation method summary

**Note:** Consider both speed and business requirements when selecting strategy

In [ ]:
print("\n" + "=" * 80)
print("📊 STRATEGY COMPARISON")
print("=" * 80 + "\n")

print("⏱️  Execution Time:")
print(f"  Strategy 1: {elapsed_s1:6.2f}s")
print(f"  Strategy 2: {elapsed_s2:6.2f}s")
print(f"  Strategy 3: {elapsed_s3:6.2f}s")
print(f"  Total:      {elapsed_s1 + elapsed_s2 + elapsed_s3:6.2f}s")

print("\n🎭 Name Characteristics:")
print("  Strategy 1: English, Gender-Aware, FL format")
print("  Strategy 2: Vietnamese, LF format")
print("  Strategy 3: Faker, FML format (with middle names)")

print("\n🔧 Generation Method:")
print("  Strategy 1: Built-in dictionary + gender field")
print("  Strategy 2: Language-specific dictionary")
print("  Strategy 3: Faker library (maximum diversity)")

## Step 6: Detailed Artifact Review

**How to use:**
- Run for comprehensive artifact inspection
- Auto-loads NEWEST files from each category
- Displays visualizations inline

**What you'll see (per strategy):**
1. **Metrics JSON**: Generation performance, name stats, length distribution
2. **Name Mapping**: Original → Synthetic name mappings
3. **Visualizations**: Name length and distribution charts (latest batch only)

**Note:** Only newest files shown to avoid confusion from multiple runs

In [ ]:
# Review all strategies
strategy_dirs = [
    ('strategy1_en_gender', 'Strategy 1: English Gender-Aware'),
    ('strategy2_multilang', 'Strategy 2: Multi-Language'),
    ('strategy3_faker', 'Strategy 3: Faker Library')
]

for dir_name, title in strategy_dirs:
    strategy_dir = task_dir / dir_name
    if not strategy_dir.exists():
        continue
        
    print("\n" + "=" * 80)
    print(f"📊 {title}")
    print("=" * 80)
    
    # 1. Metrics (NEWEST - with filtering)
    metrics_dir = strategy_dir / 'metrics'
    if metrics_dir.exists():
        metrics_files = list(metrics_dir.glob('*.json'))
        if metrics_files:
            # Exclude data_types files
            filtered = [f for f in metrics_files if not f.name.startswith("data_types")]

            if filtered:
                target_files = filtered
                print(f"✓ Found {len(filtered)} metrics file(s)")
            else:
                target_files = metrics_files
                print(f"⚠ Fallback to ALL {len(metrics_files)} file(s)")

            # Pick latest
            target_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
            latest_metrics_file = target_files[0]
            
            print(f"📄 {latest_metrics_file.name}")
            
            try:
                with open(latest_metrics_file, 'r') as f:
                    data = json.load(f)
                    metrics = data.get('metrics', {})
                
                # Performance
                if 'performance' in metrics:
                    perf = metrics['performance']
                    print(f"   Generation Time: {perf.get('generation_time', 0):.4f}s")
                    print(f"   Records/sec: {perf.get('records_per_second', 0):,}")
                
                # Name generator metrics
                if 'name_generator' in metrics:
                    ng = metrics['name_generator']
                    print(f"   Language: {ng.get('language')}")
                    print(f"   Format: {ng.get('format')}")
                    print(f"   Use Faker: {ng.get('use_faker')}")
                    
                    # Length stats
                    if 'length_stats' in ng:
                        ls = ng['length_stats']
                        print(f"   Avg Length: {ls.get('mean', 0):.1f} chars")
                        
            except Exception as e:
                print(f"   ⚠️  Error: {e}")
    
    # 2. Mapping (NEWEST)
    mapping_files = sorted(
        list(strategy_dir.glob('**/*mapping*.json')),
        key=lambda x: x.stat().st_mtime, reverse=True
    )
    if mapping_files:
        print(f"\n📄 MAPPING: {mapping_files[0].name}")
        try:
            with open(mapping_files[0], 'r') as f:
                mapping_data = json.load(f)
                mappings = mapping_data.get('mappings', {})
                if FIELD_CONFIG['name_field'] in mappings:
                    field_mappings = mappings[FIELD_CONFIG['name_field']]
                    print(f"   Total mappings: {len(field_mappings)}")
                    print(f"   Sample (first 3):")
                    for mapping in field_mappings[:3]:
                        print(f"     {mapping.get('original', 'N/A')} → {mapping.get('synthetic', 'N/A')}")
        except Exception as e:
            print(f"   ⚠️  Error: {e}")
    
    # 3. Visualizations (NEWEST BATCH)
    viz_dir = strategy_dir / 'visualizations'
    if viz_dir.exists():
        viz_files = sorted(
            list(viz_dir.glob('*.png')),
            key=lambda x: x.stat().st_mtime, reverse=True
        )
        if viz_files:
            latest_time = viz_files[0].stat().st_mtime
            latest_batch = [
                f for f in viz_files 
                if abs(f.stat().st_mtime - latest_time) < 10
            ]
            print(f"\n📊 VISUALIZATIONS: {len(latest_batch)} files")
            for viz in latest_batch[:2]:  # Show first 2
                print(f"   📈 {viz.stem.replace('_', ' ').title()}")
                try:
                    display(Image(filename=str(viz)))
                except:
                    print(f"      (Display error)")

## Step 7: Calculate Privacy Metrics

**How to use:**
- Calculate k-anonymity for original and synthetic names
- Compare privacy improvements across strategies
- Run AFTER all 3 strategies complete

**What you'll see:**
- Original data: min k, avg k, vulnerable group count
- After anonymization: improved k-values with multipliers
- Name uniqueness preservation metrics

**Privacy targets:**
- Min k ≥ 5: Basic privacy protection
- Min k ≥ 10: Strong privacy protection
- Name diversity: Maintain reasonable uniqueness

**Note:** Names combined with other quasi-identifiers for k-anonymity

In [ ]:
print("\n" + "=" * 80)
print("🔒 PRIVACY METRICS")
print("=" * 80 + "\n")

def calculate_k_anonymity(df, quasi_identifiers):
    """Calculate k-anonymity metrics."""
    existing_qis = [q for q in quasi_identifiers if q in df.columns]
    if not existing_qis:
        return None

    group_sizes = df.groupby(existing_qis).size()
    return {
        'min_k': int(group_sizes.min()),
        'avg_k': float(group_sizes.mean()),
        'vulnerable_groups': int((group_sizes < 5).sum())
    }

# Check if strategies completed
try:
    field_name = FIELD_CONFIG["name_field"]
    field_dept = "department"
    
    # Original
    k_orig = calculate_k_anonymity(df, [field_name, field_dept])
    print(f"📊 ORIGINAL DATA:")
    print(f"   Min k: {k_orig['min_k']}")
    print(f"   Avg k: {k_orig['avg_k']:.2f}")
    print(f"   Vulnerable groups: {k_orig['vulnerable_groups']}")
    
    # Name uniqueness
    print(f"\n🎭 NAME UNIQUENESS:")
    print(f"   Original: {df[field_name].nunique()} unique names")
    
    # After each strategy
    if 'result_df_s1' in locals() and result_df_s1 is not None:
        output_s1 = f"{field_name}_en_gender"
        quasi_ids_s1 = [output_s1, field_dept]
        k_s1 = calculate_k_anonymity(result_df_s1, quasi_ids_s1)
        
        print(f"\n✨ STRATEGY 1 (English Gender-Aware):")
        print(f"   Min k: {k_s1['min_k']} ({k_s1['min_k']/k_orig['min_k']:.1f}x)")
        print(f"   Unique names: {result_df_s1[output_s1].nunique()}")
    
    if 'result_df_s2' in locals() and result_df_s2 is not None:
        output_s2 = f"{field_name}_vi"
        quasi_ids_s2 = [output_s2, field_dept]
        k_s2 = calculate_k_anonymity(result_df_s2, quasi_ids_s2)
        
        print(f"\n✨ STRATEGY 2 (Multi-Language):")
        print(f"   Min k: {k_s2['min_k']} ({k_s2['min_k']/k_orig['min_k']:.1f}x)")
        print(f"   Unique names: {result_df_s2[output_s2].nunique()}")
    
    if 'result_df_s3' in locals() and result_df_s3 is not None:
        output_s3 = f"{field_name}_faker"
        quasi_ids_s3 = [output_s3, field_dept]
        k_s3 = calculate_k_anonymity(result_df_s3, quasi_ids_s3)
        
        print(f"\n✨ STRATEGY 3 (Faker Library):")
        print(f"   Min k: {k_s3['min_k']} ({k_s3['min_k']/k_orig['min_k']:.1f}x)")
        print(f"   Unique names: {result_df_s3[output_s3].nunique()}")
        
except NameError:
    print("⚠️  FIELD_CONFIG not defined!")
    print("   Please run 'Step 3: Setup Shared Environment' first")

## Step 8: Export Final Data

**How to use:**
- Export final synthetic names from all strategies
- Each strategy gets its own output folder
- Run AFTER all 3 strategies complete

**What you'll see (per strategy):**
- Available columns list
- Export confirmation (file path, row count)
- Preview of first 5 rows
- Name length statistics

**Output structure:**
```
advanced_output/
├── strategy1_en_gender/name_en_gender_data.csv
├── strategy2_multilang/name_multilang_data.csv
└── strategy3_faker/name_faker_data.csv
```

**Note:** Files include both original and synthetic names for comparison

In [ ]:
import os
from pathlib import Path

print("=" * 80)
print("📦 EXPORTING FINAL DATA FROM ALL STRATEGIES")
print("=" * 80)

# Get field config
try:
    field_name = FIELD_CONFIG["name_field"]
    gender_field = FIELD_CONFIG["gender_field"]
except NameError:
    print("❌ FIELD_CONFIG not defined! Run Step 3 first.")
    raise

# Base export directory
export_base_dir = task_dir
task_dir.mkdir(parents=True, exist_ok=True)

print(f"\n📂 Export directory: {task_dir}\n")

# ============================================================================
# STRATEGY 1: English Gender-Aware
# ============================================================================
if 'result_df_s1' in locals() and result_df_s1 is not None:
    print("=" * 80)
    print("📊 STRATEGY 1: ENGLISH GENDER-AWARE")
    print("=" * 80)
    
    s1_dir = export_base_dir / 'strategy1_en_gender'
    s1_dir.mkdir(parents=True, exist_ok=True)
    
    output_col_s1 = f"{field_name}_en_gender"
    
    if output_col_s1 in result_df_s1.columns:
        export_cols_s1 = ['employee_id', field_name, output_col_s1, 
                          gender_field, 'department']
        existing_cols_s1 = [col for col in export_cols_s1 if col in result_df_s1.columns]
        
        final_df_s1 = result_df_s1[existing_cols_s1].copy()
        
        output_path_s1 = s1_dir / 'name_en_gender_data.csv'
        try:
            final_df_s1.to_csv(output_path_s1, index=False)
            print(f"\n✅ Saved: {output_path_s1}")
            print(f"   Rows: {len(final_df_s1):,}")
        except PermissionError:
            print(f"\n⚠️  Cannot save (file may be open)")
        
        print(f"\n📊 Preview:")
        print(final_df_s1.head())
        
        # Name stats
        avg_len = final_df_s1[output_col_s1].str.len().mean()
        print(f"\n📈 Name Statistics:")
        print(f"   Unique names: {final_df_s1[output_col_s1].nunique()}")
        print(f"   Avg length: {avg_len:.1f} chars")
    else:
        print(f"\n❌ Column '{output_col_s1}' not found")
else:
    print("\n⚠️  Strategy 1 data not available")

# ============================================================================
# STRATEGY 2: Multi-Language
# ============================================================================
if 'result_df_s2' in locals() and result_df_s2 is not None:
    print("\n" + "=" * 80)
    print("📊 STRATEGY 2: MULTI-LANGUAGE")
    print("=" * 80)
    
    s2_dir = export_base_dir / 'strategy2_multilang'
    s2_dir.mkdir(parents=True, exist_ok=True)
    
    output_col_s1 = f"{field_name}_en_gender"
    output_col_s2 = f"{field_name}_vi"
    
    if output_col_s2 in result_df_s2.columns:
        export_cols_s2 = ['employee_id', field_name, output_col_s1,
                          output_col_s2, gender_field, 'department']
        existing_cols_s2 = [col for col in export_cols_s2 if col in result_df_s2.columns]
        
        final_df_s2 = result_df_s2[existing_cols_s2].copy()
        
        output_path_s2 = s2_dir / 'name_multilang_data.csv'
        try:
            final_df_s2.to_csv(output_path_s2, index=False)
            print(f"\n✅ Saved: {output_path_s2}")
            print(f"   Rows: {len(final_df_s2):,}")
        except PermissionError:
            print(f"\n⚠️  Cannot save (file may be open)")
        
        print(f"\n📊 Preview:")
        print(final_df_s2.head())
        
        # Language-specific stats
        avg_len = final_df_s2[output_col_s2].str.len().mean()
        print(f"\n📈 Vietnamese Name Stats:")
        print(f"   Avg length: {avg_len:.1f} chars")
    else:
        print(f"\n❌ Column '{output_col_s2}' not found")
else:
    print("\n⚠️  Strategy 2 data not available")

# ============================================================================
# STRATEGY 3: Faker Library
# ============================================================================
if 'result_df_s3' in locals() and result_df_s3 is not None:
    print("\n" + "=" * 80)
    print("📊 STRATEGY 3: FAKER LIBRARY")
    print("=" * 80)
    
    s3_dir = export_base_dir / 'strategy3_faker'
    s3_dir.mkdir(parents=True, exist_ok=True)
    
    output_col_s1 = f"{field_name}_en_gender"
    output_col_s2 = f"{field_name}_vi"
    output_col_s3 = f"{field_name}_faker"
    
    if output_col_s3 in result_df_s3.columns:
        export_cols_s3 = ['employee_id', field_name, output_col_s1,
                          output_col_s2, output_col_s3,
                          gender_field, 'department']
        existing_cols_s3 = [col for col in export_cols_s3 if col in result_df_s3.columns]
        
        final_df_s3 = result_df_s3[existing_cols_s3].copy()
        
        output_path_s3 = s3_dir / 'name_faker_data.csv'
        try:
            final_df_s3.to_csv(output_path_s3, index=False)
            print(f"\n✅ Saved: {output_path_s3}")
            print(f"   Rows: {len(final_df_s3):,}")
        except PermissionError:
            print(f"\n⚠️  Cannot save (file may be open)")
        
        print(f"\n📊 Preview:")
        print(final_df_s3.head())
        
        # Complexity stats
        name_parts = final_df_s3[output_col_s3].str.split().str.len()
        print(f"\n📈 Name Complexity:")
        print(f"   3-part names: {(name_parts == 3).sum()} ({(name_parts == 3).sum()/len(final_df_s3)*100:.1f}%)")
    else:
        print(f"\n❌ Column '{output_col_s3}' not found")
else:
    print("\n⚠️  Strategy 3 data not available")

# ============================================================================
# SUMMARY
# ============================================================================
print("\n" + "=" * 80)
print("✅ EXPORT COMPLETE")
print("=" * 80)
print(f"\n📂 Files saved to: {export_base_dir}")

if 'elapsed_s1' in locals():
    total_time = elapsed_s1 + elapsed_s2 + elapsed_s3
    print(f"⏱️  Total time: {total_time:.2f}s")

## 🎯 Summary

**Accomplished:**
- ✅ 3 name generation strategies implemented and compared
- ✅ Privacy metrics calculated (k-anonymity, uniqueness)
- ✅ Performance benchmarking completed
- ✅ Production-ready artifacts generated

**Key takeaways:**
- Gender-aware generation maintains demographic consistency
- Multi-language support enables global dataset protection
- Faker library provides maximum name diversity
- Name format affects cultural appropriateness (FL vs LF)
- Mapping consistency ensures deterministic generation

**Strategy recommendations:**
- **Use Strategy 1** when: Gender consistency critical, English-speaking context
- **Use Strategy 2** when: Multi-cultural datasets, language-specific requirements
- **Use Strategy 3** when: Maximum realism needed, complex name formats required

**Next steps:**
- Test with your own datasets
- Customize name dictionaries for specific regions
- Adjust format patterns (FML, LFM, F_L, etc.)
- Deploy to production environment

**Resources:**
- 📖 [PAMOLA Documentation](../../docs/)
- 🐙 [GitHub Repository](https://github.com/DGT-Network/PAMOLA)